In [1]:
import dataset_loader
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

from dataset_loader import (
    load_metadata,
    get_subject_list,
    load_subject_imu,
    get_dataset_path
)

In [2]:
metadata = load_metadata()
subjects = get_subject_list()

print("Dataset Loaded Successfully")

print(f"Metadata Entries : {len(metadata)}")
print(f"Subject Folders  : {len(subjects)}")
print("Metadata Columns")
print(metadata.columns.tolist())

Dataset Loaded Successfully
Metadata Entries : 60
Subject Folders  : 67
Metadata Columns
['Participant ID', 'Gender', 'Age', 'Height (m)', 'Weight (kg)', 'Fat %', 'BMI', 'SpO2_baseline(%)', 'HR_baseline(bpm)', 'HR step test(bpm)']


In [3]:
# Convert Metadata IDs to Folder Names
metadata_subjects = set(
    metadata["Participant ID"].apply(
        lambda x: f"Subject{int(x):02d}"
    )
)

folder_subjects = set(subjects)

print(f"Metadata Subjects : {len(metadata_subjects)}")
print(f"Folder Subjects   : {len(folder_subjects)}")

Metadata Subjects : 60
Folder Subjects   : 67


In [4]:
# Compare Metadata with Folder Names
matched_subjects = sorted(
    metadata_subjects.intersection(folder_subjects)
)

extra_folders = sorted(
    folder_subjects - metadata_subjects
)

missing_folders = sorted(
    metadata_subjects - folder_subjects
)

print("Dataset Consistency Check")

print(f"Matched Subjects : {len(matched_subjects)}")
print(f"Extra Folders    : {len(extra_folders)}")
print(f"Missing Folders  : {len(missing_folders)}")

print("\nExtra Folder Names")
print(extra_folders)

print("\nMissing Folder Names")
print(missing_folders)

Dataset Consistency Check
Matched Subjects : 60
Extra Folders    : 7
Missing Folders  : 0

Extra Folder Names
['Subject20', 'Subject29', 'Subject30', 'Subject35', 'Subject43', 'Subject61', 'Subject65']

Missing Folder Names
[]


In [5]:
# Inspect Extra Subject Folders


dataset_path = get_dataset_path()

summary = []

for subject in extra_folders:

    subject_folder = dataset_path / subject

    files = list(subject_folder.iterdir())

    csv_files = list(subject_folder.glob("*.csv"))

    if len(csv_files) > 0:

        df = pd.read_csv(csv_files[0])

        summary.append({
            "Subject": subject,
            "Files": len(files),
            "CSV Name": csv_files[0].name,
            "Rows": df.shape[0],
            "Columns": df.shape[1]
        })

    else:

        summary.append({
            "Subject": subject,
            "Files": len(files),
            "CSV Name": "No CSV",
            "Rows": 0,
            "Columns": 0
        })

summary_df = pd.DataFrame(summary)

summary_df

,Subject,Files,CSV Name,Rows,Columns
0,Subject20,1,IMUSubject20.csv,110553,33
1,Subject29,1,IMUSubject29.csv,85705,33
2,Subject30,1,IMUSubject30.csv,107825,33
3,Subject35,1,IMUSubject35.csv,107622,33
4,Subject43,1,IMUSubject43.csv,109519,32
5,Subject61,1,IMUSubject61.csv,107420,33
6,Subject65,1,IMUSubject65.csv,107810,33


In [6]:
# Check IMU File Structure for All Subjects

structure = []

for subject in subjects:

    df = load_subject_imu(subject)

    structure.append({
        "Subject": subject,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column Names": tuple(df.columns)
    })

structure_df = pd.DataFrame(structure)

print("Unique column counts:")
print(structure_df["Columns"].value_counts())

structure_df.head()

Unique column counts:
Columns
33    50
32    17
Name: count, dtype: int64


,Subject,Rows,Columns,Column Names
0,Subject01,110740,32,"(epoch, timestamp_unified, q_w_chest, q_x_ches..."
1,Subject02,110563,32,"(epoch, timestamp_unified, q_w_chest, q_x_ches..."
2,Subject03,98902,32,"(timestamp, time, q_w_chest, q_x_chest, q_y_ch..."
3,Subject04,106672,32,"(timestamp, time, q_w_chest, q_x_chest, q_y_ch..."
4,Subject05,105411,33,"(timestamp, time, q_w_chest, q_x_chest, q_y_ch..."


In [7]:
# Subjects grouped by number of columns

structure_df.groupby("Columns")["Subject"].apply(list)

Columns
32    [Subject01, Subject02, Subject03, Subject04, S...
33    [Subject05, Subject06, Subject10, Subject12, S...
Name: Subject, dtype: object

In [8]:
# Compare Column Names
subject_32 = load_subject_imu("Subject01")
subject_33 = load_subject_imu("Subject05")

columns_32 = set(subject_32.columns)
columns_33 = set(subject_33.columns)

print("Columns only in Subject01 (32-column)")
print(columns_32 - columns_33)

print("\n")

print("Columns only in Subject05 (33-column)")

print(columns_33 - columns_32)

Columns only in Subject01 (32-column)
{'epoch'}


Columns only in Subject05 (33-column)
{'timestamp', 'time'}


In [9]:
# Verify Sensor Columns Across All Subjects

reference_columns = None
different_subjects = []

for subject in subjects:

    sensor_df = load_subject_imu(subject, sensors_only=True)

    columns = list(sensor_df.columns)

    if reference_columns is None:
        reference_columns = columns

    elif columns != reference_columns:
        different_subjects.append(subject)

print("Sensor Column Consistency Check")

print(f"Reference Sensor Columns : {len(reference_columns)}")
print(f"Subjects with Different Sensor Columns : {len(different_subjects)}")

if len(different_subjects) > 0:
    print(different_subjects)
else:
    print("All subjects have identical sensor columns.")

Sensor Column Consistency Check
Reference Sensor Columns : 30
Subjects with Different Sensor Columns : 0
All subjects have identical sensor columns.


In [11]:
# Section 2 : Load Sensor Data
sensor_data = {}

for subject in subjects:

    sensor_data[subject] = load_subject_imu(
        subject,
        sensors_only=True
    )

print(f"Loaded sensor data for {len(sensor_data)} subjects.")

Loaded sensor data for 67 subjects.


In [12]:

# Missing Value Analysis

missing_summary = []

for subject, df in sensor_data.items():

    missing_summary.append({
        "Subject": subject,
        "Missing Values": df.isnull().sum().sum()
    })

missing_df = pd.DataFrame(missing_summary)

missing_df

subjects_with_missing = (
    missing_df["Missing Values"] > 0
).sum()

# Overall Missing Value Summary
print("Missing Value Summary")

print(f"Total Subjects              : {len(missing_df)}")
print(f"Subjects with Missing Values: {subjects_with_missing}")

if subjects_with_missing == 0:
    print("\n No missing values found.")
else:
    print("\n Missing values detected.")

Missing Value Summary
Total Subjects              : 67
Subjects with Missing Values: 0

 No missing values found.


In [29]:
# Data Type Verification

dtype_summary = []

for subject, df in sensor_data.items():

    unique_dtypes = df.dtypes.unique()

    dtype_summary.append({
        "Subject": subject,
        "Data Types": list(unique_dtypes)
    })

dtype_df = pd.DataFrame(dtype_summary)

dtype_df.head()

# Unique Data Types Across Dataset
all_dtypes = set()

for df in sensor_data.values():
    all_dtypes.update(df.dtypes.astype(str).tolist())


print("Unique Data Types")
for dtype in sorted(all_dtypes):
    print(dtype)

Unique Data Types
float64


In [13]:
# Create Subject List
metadata_subjects = metadata["Participant ID"].tolist()

print(f"Number of Subjects : {len(metadata_subjects)}")
print(metadata_subjects[:5])

# Subject-wise Train/Test Split
train_subjects, test_subjects = train_test_split(
    metadata_subjects,
    test_size=0.20,
    random_state=42,
    shuffle=True
)
# Verify Split
print("Train-Test Split")
print(f"Training Subjects : {len(train_subjects)}")
print(f"Testing Subjects  : {len(test_subjects)}")

print("\nFirst 5 Training Subjects")
print(train_subjects[:5])

print("\nFirst 5 Testing Subjects")
print(test_subjects[:5])

Number of Subjects : 60
[1, 2, 3, 4, 5]
Train-Test Split
Training Subjects : 48
Testing Subjects  : 12

First 5 Training Subjects
[36, 4, 58, 18, 9]

First 5 Testing Subjects
[1, 6, 41, 51, 14]


In [14]:
# Create Train and Test Sensor Data
train_sensor_data = {}
test_sensor_data = {}

# Training subjects
for subject_id in train_subjects:
    subject_name = f"Subject{subject_id:02d}"
    train_sensor_data[subject_name] = load_subject_imu(
        subject_name,
        sensors_only=True
    )

# Testing subjects
for subject_id in test_subjects:

    subject_name = f"Subject{subject_id:02d}"
    test_sensor_data[subject_name] = load_subject_imu(
        subject_name,
        sensors_only=True
    )

# Verify Train/Test Data

print("Training Data")

print(f"Training Subjects Loaded : {len(train_sensor_data)}")

print("\nExample:")
first_train = list(train_sensor_data.keys())[0]
print(first_train)
print(train_sensor_data[first_train].shape)

print("Testing Data")

print(f"Testing Subjects Loaded : {len(test_sensor_data)}")

print("\nExample:")
first_test = list(test_sensor_data.keys())[0]
print(first_test)
print(test_sensor_data[first_test].shape)

Training Data
Training Subjects Loaded : 48

Example:
Subject36
(107760, 30)
Testing Data
Testing Subjects Loaded : 12

Example:
Subject01
(110740, 30)


In [15]:
# Combine All Training Sensor Data
train_df = pd.concat(
    train_sensor_data.values(),
    axis=0,
    ignore_index=True
)


print("Combined Training Dataset")

print(f"Shape : {train_df.shape}")

# Compute Training Statistics
train_mean = train_df.mean(axis=0)

train_std = train_df.std(axis=0)

print("Training Statistics")

print("\nMean")
print(train_mean)

print("\nStandard Deviation")
print(train_std)

# Verify Statistics
print(f"Number of Features : {len(train_mean)}")
print(f"Mean Shape : {train_mean.shape}")
print(f"Std Shape : {train_std.shape}")

Combined Training Dataset
Shape : (5130454, 30)
Training Statistics

Mean
q_w_chest         0.543450
q_x_chest        -0.161701
q_y_chest         0.392619
q_z_chest         0.080390
q_w_left_hand     0.617846
q_x_left_hand    -0.191809
q_y_left_hand    -0.383483
q_z_left_hand    -0.006939
q_w_right_knee    0.695752
q_x_right_knee   -0.041916
q_y_right_knee    0.321602
q_z_right_knee    0.174436
a_x_chest        -0.916820
a_y_chest        -0.031342
a_z_chest         0.115047
g_x_chest        -0.512608
g_y_chest         0.294239
g_z_chest         0.012367
a_x_left_knee    -0.656492
a_y_left_knee    -0.060109
a_z_left_knee     0.491915
g_x_left_knee    -1.194867
g_y_left_knee     0.218803
g_z_left_knee    -0.929458
a_x_right_hand    0.623925
a_y_right_hand    0.234627
a_z_right_hand    0.322048
g_x_right_hand   -0.314993
g_y_right_hand   -0.288738
g_z_right_hand    0.860488
dtype: float64

Standard Deviation
q_w_chest          0.230989
q_x_chest          0.399574
q_y_chest          0.2851

In [16]:
# Standardize Training Data
train_sensor_standardized = {}

for subject, df in train_sensor_data.items():

    standardized_df = (df - train_mean) / train_std

    train_sensor_standardized[subject] = standardized_df

print(f"Training subjects standardized : {len(train_sensor_standardized)}")

# Standardize Testing Data
test_sensor_standardized = {}

for subject, df in test_sensor_data.items():

    standardized_df = (df - train_mean) / train_std

    test_sensor_standardized[subject] = standardized_df

print(f"Testing subjects standardized : {len(test_sensor_standardized)}")
# Verify Standardization
subject = list(train_sensor_standardized.keys())[0]

print(subject)

train_sensor_standardized[subject].head()

Training subjects standardized : 48
Testing subjects standardized : 12
Subject36


,q_w_chest,q_x_chest,q_y_chest,q_z_chest,q_w_left_hand,q_x_left_hand,q_y_left_hand,q_z_left_hand,q_w_right_knee,q_x_right_knee,...,a_z_left_knee,g_x_left_knee,g_y_left_knee,g_z_left_knee,a_x_right_hand,a_y_right_hand,a_z_right_hand,g_x_right_hand,g_y_right_hand,g_z_right_hand
0,1.314134,0.309581,0.481843,-0.205957,1.145042,-0.031277,0.013833,-0.136148,1.454269,0.274586,...,0.804279,0.031480,-0.012749,0.031882,0.669796,-0.030906,0.133848,-0.023165,0.004780,-0.036273
1,1.314134,0.309581,0.485350,-0.205957,1.145042,-0.027874,0.013833,-0.140838,1.454269,0.271107,...,0.795867,0.032664,-0.002261,0.084870,0.667298,-0.034624,0.126119,-0.039273,-0.013397,-0.033747
2,1.309804,0.309581,0.485350,-0.205957,1.140998,-0.031277,0.010747,-0.143183,1.454269,0.271107,...,0.789138,0.036216,0.004731,0.118550,0.689785,-0.032765,0.144154,-0.045130,-0.031558,-0.025343
3,1.314134,0.309581,0.485350,-0.205957,1.136954,-0.031277,0.004575,-0.145528,1.454269,0.274586,...,0.792503,0.038584,0.011723,0.118550,0.689785,-0.034624,0.141578,-0.045130,-0.033578,-0.030380
4,1.309804,0.309581,0.485350,-0.205957,1.132910,-0.031277,-0.004683,-0.145528,1.454269,0.274586,...,0.794185,0.024376,0.022197,0.111324,0.669796,-0.045776,0.133848,-0.040737,-0.004309,-0.038798


In [17]:
# Sliding Window Function
def create_sliding_windows(dataframe, window_size=128, stride=10):
    windows = []

    for start in range(
        0,
        len(dataframe) - window_size + 1,
        stride
    ):

        end = start + window_size

        window = dataframe.iloc[start:end].values

        windows.append(window)

    return np.array(windows)

# Test Sliding Window
subject = list(train_sensor_standardized.keys())[0]

subject_data = train_sensor_standardized[subject]

windows = create_sliding_windows(
    subject_data,
    window_size=128,
    stride=10
)

print("Sliding Window Test")
print("Subject :", subject)
print("Original Shape :", subject_data.shape)
print("Window Shape :", windows.shape)

Sliding Window Test
Subject : Subject36
Original Shape : (107760, 30)
Window Shape : (10764, 128, 30)


In [18]:
# Create Sliding Windows for Training Subjects
train_windows = {}

for subject, df in train_sensor_standardized.items():

    train_windows[subject] = create_sliding_windows(
        df,
        window_size=128,
        stride=10
    )

print(f"Training subjects processed : {len(train_windows)}")

# Create Sliding Windows for Testing Subjects
test_windows = {}

for subject, df in test_sensor_standardized.items():

    test_windows[subject] = create_sliding_windows(
        df,
        window_size=128,
        stride=10
    )

print(f"Testing subjects processed : {len(test_windows)}")

# Verify Window Shapes
subject = list(train_windows.keys())[0]

print("Subject :", subject)

print("Window Shape :", train_windows[subject].shape)

Training subjects processed : 48
Testing subjects processed : 12
Subject : Subject36
Window Shape : (10764, 128, 30)


In [19]:
print("value:",metadata.columns)

value: Index(['Participant ID', 'Gender', 'Age', 'Height (m)', 'Weight (kg)', 'Fat %',
       'BMI', 'SpO2_baseline(%)', 'HR_baseline(bpm)', 'HR step test(bpm)'],
      dtype='object')


In [20]:
subject = "Subject01"

df = load_subject_imu(subject, sensors_only=False)

print(df.columns)
print(df.head())

Index(['epoch', 'timestamp_unified', 'q_w_chest', 'q_x_chest', 'q_y_chest',
       'q_z_chest', 'q_w_left_hand', 'q_x_left_hand', 'q_y_left_hand',
       'q_z_left_hand', 'q_w_right_knee', 'q_x_right_knee', 'q_y_right_knee',
       'q_z_right_knee', 'a_x_chest', 'a_y_chest', 'a_z_chest', 'g_x_chest',
       'g_y_chest', 'g_z_chest', 'a_x_left_knee', 'a_y_left_knee',
       'a_z_left_knee', 'g_x_left_knee', 'g_y_left_knee', 'g_z_left_knee',
       'a_x_right_hand', 'a_y_right_hand', 'a_z_right_hand', 'g_x_right_hand',
       'g_y_right_hand', 'g_z_right_hand'],
      dtype='object')
          epoch    timestamp_unified  q_w_chest  q_x_chest  q_y_chest  \
0  1.740000e+12  09:48:30 27/02/2025      0.840     -0.005      0.542   
1  1.740000e+12  09:48:30 27/02/2025      0.840     -0.005      0.543   
2  1.740000e+12  09:48:30 27/02/2025      0.840     -0.004      0.543   
3  1.740000e+12  09:48:30 27/02/2025      0.840     -0.004      0.543   
4  1.740000e+12  09:48:30 27/02/2025      0.83

In [21]:
bio = pd.read_csv(
    r"C:\PrivDiffuser\datasets\DatasetIMUandBIOMARKERS\Subject01\BiomarkersSubject01.csv"
)

print(bio.columns)
print(bio.head())

Index(['SpO2', 'HR', 'ActivityLabel'], dtype='object')
   SpO2     HR  ActivityLabel
0  97.0  100.0            0.0
1  96.0  100.0            0.0
2  96.0  100.0            0.0
3  95.0  100.0            0.0
4  95.0  103.0            0.0
